# 09 · 관계 색인과 짧은 탐색 계획 가설 검증

**가설:** 문서 내용을 매번 넓게 판단하는 대신, 사전에 원문에 연결된 관계를 저장하고 짧은 관계 경로를 판단하면, 같은 근거 품질에서 판단 입력·응답 시간·생성기 입력을 줄일 수 있다.

이 노트북은 **개발 실험**이다. 기존 05–08과 결과는 보존한다. A/B는 같은 RetroMAE 후보 50개에서 원문 최대 5개를 전달한다. 새 C/D는 별도 관계 색인에서 후보를 만든다. C의 원문/인용 비교와 모델 없는 대조군을 함께 둔다. 코퍼스는 정답 근거를 포함하도록 준비된 축소 KILT 코퍼스이며 전체 Wikipedia 평가가 아니다.


In [ ]:
from pathlib import Path
import sys, os, math, json, warnings
warnings.filterwarnings("ignore", message="IProgress not found.*", module="tqdm.auto")
import pandas as pd
import pyarrow  # Windows: before Torch.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, ticker
from IPython.display import display, Markdown
ROOT = Path.cwd() if (Path.cwd()/"rag_baselines").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from rag_baselines.data import read_json, read_jsonl, write_json, file_sha256
from rag_baselines.plan_experiment import METHODS, run_plan_experiment, load_plan_report, validate_settings, run_packing_replay, load_packing_report
from rag_baselines.evaluation import notebook_source_sha256
SETTINGS = read_json(ROOT/"config/plan_routing.json")
validate_settings(SETTINGS)
font = Path("C:/Windows/Fonts/malgun.ttf")
if font.exists():
    font_manager.fontManager.addfont(str(font))
    plt.rcParams["font.family"] = font_manager.FontProperties(fname=str(font)).get_name()
plt.rcParams.update({"font.size":11,"axes.unicode_minus":False,"axes.spines.top":False,"axes.spines.right":False,
    "figure.facecolor":"white","axes.facecolor":"white","text.color":"#263746","axes.labelcolor":"#263746"})
NAMES = ["A\n기본 RAG","B\n재정렬 RAG","C\n계획·원문","C\n계획·인용","D\n계획·인용","대조군\n코드·인용"]
COLORS = ["#416D95","#BE7D32","#537D59","#83A184","#8565A8","#88939C"]
display(pd.DataFrame([{"문항/과제":SETTINGS["questions_per_task"],"총 문항":2*SETTINGS["questions_per_task"],
    "반복":SETTINGS["repeats"],"최대 후보 계획":SETTINGS["search"]["max_candidate_plans"],
    "선택 계획":SETTINGS["search"]["selected_plans"],"원문 최대":SETTINGS["search"]["evidence_k"],
    "사전 추출 문단/요청":SETTINGS["extraction"]["passages_per_request"]}]))


## 수식과 반증 조건

원문 집합을 $\mathcal D$, 색인의 관계 수를 $F$, 질문 $q$의 이름 일치 후보를 $E_q$, 최종 후보 계획을 $\mathcal P_q$라 둔다. 각 계획은 **원문에 연결된 관계 1~2개**다. 코드가 이름의 역색인과 관계 연결을 이용해 계획을 만들며, 후보 상한 $P_{\max}$를 적용한다.

$$p_\theta(\pi,q)=\mathrm{Noul}_\theta(q,\mathrm{card}(\pi)),\quad
\Pi_q=\mathrm{TopK}_{\pi\in\mathcal P_q}p_\theta(\pi,q),\quad
D_q=\bigcup_{\pi\in\Pi_q}\mathrm{sources}(\pi).$$

$p_\theta$는 보정된 정확도 확률로 간주하지 않고 **순위를 만드는 판단값**으로 쓴다. 여러 계획을 독립 평가한다. $D_q$의 상한도 5개다. 모델 없는 대조군은 동일한 $\mathcal P_q$를 코드의 단어 일치 점수로 선택한다.

$$T(q)=T_{\mathrm{entity}}+T_{\mathrm{paths}}+T_{\mathrm{judge}}+T_{\mathrm{pack}}+T_{\mathrm{generator}},$$
$$I_{\mathrm{judge}}(q)=\sum_j I_j,\quad
C_{\mathrm{online}}=C_{\mathrm{judge}}+C_{\mathrm{generator}},\quad
C_{\mathrm{amortized}}(Q)=C_{\mathrm{prep}}/Q+\overline{C_{\mathrm{online}}}.$$

$Q$는 색인을 재사용하는 질의 수다. 사전 비용은 별도로 보고하고 임의의 $Q$에서 무료로 취급하지 않는다. 이름 찾기와 경로 열거의 비용은 자료량에 의존하므로 **상수 시간 검색을 주장하지 않는다**. 서로 다른 모델의 토큰을 합쳐 동일한 연산량으로 해석하지 않는다.

정답 근거 ID 집합을 $G_q$라 할 때:
$$R_{\mathrm{candidate}}=\frac1{|Q|}\sum_q\mathbf1[G_q\subseteq\cup_{\pi\in\mathcal P_q}\mathrm{sources}(\pi)],\qquad
R_{\mathrm{returned}}=\frac1{|Q|}\sum_q\mathbf1[G_q\subseteq D_q].$$

이것은 **원문 ID 확보율**이다. 일부 인용에 필요한 모든 내용이 남았다는 보장은 아니다. 인용 손실은 동일 근거 ID의 C 원문/인용 답변 EM과, 정답 별칭 포함 여부를 함께 보되 후자는 진단용 지표로만 쓴다.

**반증 기준:** 후보 생성에서 근거가 빠지거나, 판단모델이 코드 대조군보다 좋은 근거를 고르지 못하거나, 인용으로 줄였을 때 답변 품질이 떨어지면 해당 부분의 가설은 지지되지 않는다. 비용·시간 감소만으로 성공을 선언하지 않는다.

선택 원문이 후보 원문의 부분집합인 이 구현에서, 후보 원문 집합을 $I_q$라 하면:

$$\mathbf 1[G_q \subseteq E_\theta(q)] \le \mathbf 1[G_q \subseteq I_q], \qquad R_{\mathrm{returned}} \le R_{\mathrm{candidate}}.$$

따라서 후보 누락은 선택기 개선만으로 복구할 수 없다. 이 상한은 **필수 원문 ID 확보율**의 상한이며 답변 EM의 상한이 아니다. 다른 근거나 생성기의 사전 지식으로 정답을 맞힐 가능성을 같은 것으로 취급하지 않는다.


## 실행과 저장

모든 설정은 `config/plan_routing.json`에서 바꾼다. 실제 실행은 새 원시 로그를 남긴다. 색인 추출은 질문·정답 없이 문단만 입력하며 완료된 동일 색인은 재사용한다. 중간 추출 결과는 배치별로 저장된다. 각 작업 스레드는 독립 HTTP 로그를 사용한다.

`RUN_PLAN_EXPERIMENT=True`로 두면 유료 준비·판단·생성 호출을 포함한다. 기본값 False는 완료된 결과를 검증하고 읽는 분석 모드다. 원문/인용 비교에서도 지연 측정을 위해 검색을 새로 실행한다. 실시간 선택의 변동을 기록한다. 인용만의 효과는 별도의 고정 검색 결과 품질 재생에서 비교한다.


In [ ]:
RUN_PLAN_EXPERIMENT = False
RUN_PLAN_EXPERIMENT = RUN_PLAN_EXPERIMENT or os.environ.get("DECISION_LLM_RUN_PLANS") == "1"
SYSTEM_RUN = None  # "results/plans_..."로 과거 실행 지정 가능
if RUN_PLAN_EXPERIMENT:
    from collections import deque
    from IPython.display import clear_output
    messages = deque(maxlen=8)
    def progress(message):
        messages.append(message)
        clear_output(wait=True)
        print("09 진행 중 · 배치/호출/문항별 로그가 저장됩니다.")
        print("\n".join(messages))
    RUN_DIR = run_plan_experiment(ROOT,SETTINGS,progress=progress)
elif SYSTEM_RUN:
    RUN_DIR = ROOT/SYSTEM_RUN
else:
    pointer = ROOT/"results/current_plan_run.txt"
    RUN_DIR = ROOT/pointer.read_text(encoding="utf-8").strip() if pointer.exists() else None
if RUN_DIR is None:
    print("완료된 09 결과가 없습니다. RUN_PLAN_EXPERIMENT=True로 실행하세요.")
else:
    df,table,protocol,manifest = load_plan_report(ROOT,RUN_DIR)
    records = read_jsonl(RUN_DIR/"system.jsonl")
    ANALYSIS = RUN_DIR/"analysis"
    ANALYSIS.mkdir(exist_ok=True)
    print("검증한 실행:",RUN_DIR)
    print("범위: 개발 파일럿 ·",len(protocol["question_ids"]),"문항 ·",len(df),"실제 생성 관측")


## 선택을 고정한 인용 품질 검증

실제 검색을 재호출하면 같은 후보에서도 계획 순위·선택·원문 순서가 달라질 수 있다. 아래는 C 원문 실행에서 저장된 **동일 사실·동일 원문 ID·동일 순서**를 사용한 별도 인용 생성이다. 실시간 검색을 다시 하지 않으므로 답변 품질·실제 GPT 입력을 비교하며 전체 응답 지연으로 해석하지 않는다. 기본은 저장 결과 읽기이며 실행 스위치 또는 `DECISION_LLM_RUN_PACKING=1`을 켜면 설정의 최대 40회 생성 호출을 사용한다. 전체 실험을 새로 실행하면 240회 본 비교와 40회 고정 품질 재생의 한도를 따로 적용한다.


In [ ]:
RUN_FIXED_PACKING = False
RUN_FIXED_PACKING = RUN_FIXED_PACKING or RUN_PLAN_EXPERIMENT or os.environ.get("DECISION_LLM_RUN_PACKING") == "1"
PACKING_DIR = None
if RUN_DIR is not None:
    variability = pd.DataFrame(manifest["packing_variability"])
    variability.to_csv(ANALYSIS/"live_selection_variability.csv",index=False)
    print("실시간 C 원문/인용: 계획 또는 순서가 달라진 관측",int((~variability.same_plan_order).sum()),"/",len(variability))
    print("원문 순서와 사실 집합까지 동일한 실시간 관측",int(variability.packing_controlled.sum()),"/",len(variability))
    if RUN_FIXED_PACKING:
        PACKING_DIR=run_packing_replay(ROOT,RUN_DIR,max_calls=SETTINGS.get("max_packing_replay_calls",40),progress=print)
    elif (RUN_DIR/"current_packing_run.txt").exists():
        PACKING_DIR=ROOT/(RUN_DIR/"current_packing_run.txt").read_text(encoding="utf-8").strip()
    if PACKING_DIR is not None:
        packing_df,packing_protocol,packing_manifest=load_packing_report(ROOT,PACKING_DIR)
        packing_df.to_csv(ANALYSIS/"fixed_source_packing.csv",index=False)
        fixed=pd.DataFrame({"condition":["C original","C same sources, quotes"],
            "EM":[packing_df.original_em.mean(),packing_df.packed_em.mean()],
            "GPT_input":[packing_df.original_input_tokens.mean(),packing_df.packed_input_tokens.mean()]})
        display(fixed.round(3))
        print("고정 인용에서 좋아진/나빠진/같은 답변 수:",int((packing_df.packed_em>packing_df.original_em).sum()),
            int((packing_df.packed_em<packing_df.original_em).sum()),int((packing_df.packed_em==packing_df.original_em).sum()))
        fig,axes=plt.subplots(1,2,figsize=(11,4.7),layout="constrained")
        for ax,col,title in [(axes[0],"EM","동일 근거·순서의 답변 EM"),(axes[1],"GPT_input","동일 근거의 실제 GPT 입력")]:
            bars=ax.bar([0,1],fixed[col],color=["#537D59","#83A184"])
            bars[1].set_hatch("///");bars[1].set_edgecolor("white")
            ax.bar_label(bars,labels=[f"{v:.1%}" if col=="EM" else f"{v:.0f}" for v in fixed[col]],padding=4)
            ax.set_xticks([0,1],["C 원문","C 고정 인용"]);ax.set_title(title)
            ax.set_ylim(0,1.12 if col=="EM" else fixed[col].max()*1.2)
            if col=="EM":ax.yaxis.set_major_formatter(ticker.PercentFormatter(1))
            ax.grid(axis="y",color="#E3E8EC");ax.set_axisbelow(True)
        fig.suptitle("캐시 검색 결과 품질 재생 · 전체 지연 측정과 분리",fontsize=12)
        fig.savefig(ANALYSIS/"fixed_source_packing.png",dpi=160,bbox_inches="tight");plt.show()


## 같은 질문에서의 실제 결과

A/B는 매번 벡터 검색을 수행했다. C/D/코드 대조군은 매번 같은 규약으로 관계 후보를 재생성했다. 다음 품질 지표는 **저장된 검색 결과를 오프라인 평가**한 값이며, 노트북 재분석 시간이 온라인 지연 측정에 추가되지 않는다. 온라인 지연은 실제 실행의 벽시계 시간으로 진단 기록 작업을 포함한다.


In [ ]:
if RUN_DIR is not None:
    columns = ["questions","answer_em","answer_f1","complete_candidate_hit","complete_evidence_hit",
        "online_seconds","retrieval_seconds","llm_input_tokens","llm_output_tokens","decision_input_tokens",
        "evidence_count","online_api_cost_usd"]
    view = table[columns].copy()
    view.index = [n.replace("\n"," · ") for n in NAMES]
    display(view.round(5))
    view.to_csv(ANALYSIS/"comparison.csv")


In [ ]:
if RUN_DIR is not None:
    x = np.arange(len(METHODS))
    fig,axes = plt.subplots(3,2,figsize=(14.2,13.6),layout="constrained")
    specifications = [("answer_em","실제 답변 정확도 · EM","비율"),
        ("complete_evidence_hit","필수 원문 ID 확보율","비율"),
        ("online_seconds","실제 전체 응답 시간","초 / 문항"),
        ("llm_input_tokens","공통 GPT 실제 입력","API 청구 입력 토큰 / 문항"),
        ("decision_input_tokens","판단모델이 실제 처리한 입력","모델별 토큰 / 문항"),
        ("llm_output_tokens","공통 GPT 실제 출력","API 출력 토큰 / 문항")]
    for ax,(column,title,unit) in zip(axes.flat,specifications):
        values = table[column].to_numpy()
        errors = None
        if column == "answer_em" and protocol["settings"]["repeats"] == 1:
            n = len(protocol["question_ids"]); z=1.96
            center=(values+z*z/(2*n))/(1+z*z/n)
            half=z*np.sqrt(values*(1-values)/n+z*z/(4*n*n))/(1+z*z/n)
            errors=np.vstack([values-(center-half),(center+half)-values])
        bars = ax.bar(x,values,color=COLORS,yerr=errors,capsize=4)
        bars[3].set_hatch("///"); bars[3].set_edgecolor("white")
        labels = [f"{v:.1%}" if unit=="비율" else f"{v:,.2f}" if unit.startswith("초") else f"{v:,.0f}" for v in values]
        ax.bar_label(bars,labels=labels,padding=4,fontsize=10)
        ax.set_xticks(x,NAMES,fontsize=10)
        ax.set_title(title); ax.set_ylabel(unit)
        ax.set_ylim(0,1.15 if unit=="비율" else max(values.max()*1.28,1))
        if unit=="비율": ax.yaxis.set_major_formatter(ticker.PercentFormatter(1))
        ax.grid(axis="y",color="#E3E8EC",linewidth=.7); ax.set_axisbelow(True)
    fig.suptitle(f"09 · 관계 계획 파일럿 · {len(protocol['question_ids'])}문항 · {protocol['fact_index']['passages']:,}문단 · 반복 {protocol['settings']['repeats']}",fontsize=14)
    fig.savefig(ANALYSIS/"plan_comparison.png",dpi=160,bbox_inches="tight")
    plt.show()
    display(Markdown("EM 오차 막대: 반복 1회일 때 문항 단위 Wilson 95% 구간. 판단 토큰은 Jev/Laya 각각의 토크나이저 값이며 상호 연산량 비교용이 아니다."))


## 어디에서 근거가 사라졌는가

1. **적재:** 정답 문단이 색인에 포함되는가? 원문에 사실 하나가 있다는 것과 질문에 필요한 관계가 있다는 것은 다르다.
2. **후보 생성:** 이름 일치·문자 정규화·1~2단계 연결·후보 상한 때문에 정답 문단이 후보 밖으로 밀렸는가?
3. **계획 판단:** 후보에는 있었지만 선택한 근거에 없었는가?
4. **전달/생성:** 근거 ID가 같은 C 원문/인용에서 답변이 달라지는가? EM만으로 생성 실패 원인을 단정하지 않는다.


In [ ]:
if RUN_DIR is not None:
    plan_methods = ["C_plan","C_pack","D_pack","P_code"]
    diagnostics = table.loc[plan_methods,["indexed_gold_passages_complete","seed_found","candidate_plans",
        "complete_candidate_hit","complete_evidence_hit","answer_alias_in_generation_view","answer_em"]]
    display(diagnostics.round(4))
    diagnostics.to_csv(ANALYSIS/"failure_stages.csv")
    segments=df.groupby(["task","method"]).agg(questions=("question_id","nunique"),EM=("answer_em","mean"),
        candidate_complete=("complete_candidate_hit","mean"),returned_complete=("complete_evidence_hit","mean"),
        total_seconds=("online_seconds","mean"),GPT_input=("llm_input_tokens","mean"))
    display(segments.round(4)); segments.to_csv(ANALYSIS/"task_results.csv")
    failures=df[df.method.isin(plan_methods)].copy()
    failures["failure_stage"]=np.select([
        failures.indexed_gold_passages_complete==0,failures.complete_candidate_hit==0,
        failures.complete_evidence_hit==0,failures.answer_em==0],
        ["no_fact_in_gold_passage","candidate_generation","plan_selection","packing_or_generation"],default="answer_correct")
    failures.to_csv(ANALYSIS/"question_failures.csv",index=False)
    display(failures.groupby(["method","failure_stage"]).size().unstack(fill_value=0))


In [ ]:
if RUN_DIR is not None:
    paired = df.groupby(["question_id","method"]).answer_em.mean().unstack()
    compared=[]
    rng=np.random.default_rng(20261004)
    for candidate,baseline in [("C_plan","B"),("C_pack","B"),("D_pack","B"),("C_pack","P_code"),("D_pack","P_code"),("C_pack","C_plan")]:
        delta=(paired[candidate]-paired[baseline]).to_numpy()
        boot=rng.choice(delta,size=(5000,len(delta))).mean(axis=1)
        lo,hi=np.quantile(boot,[.025,.975])
        plus,minus=int((delta>0).sum()),int((delta<0).sum())
        discordant=plus+minus
        p=1. if not discordant else min(1.,2*sum(math.comb(discordant,k) for k in range(min(plus,minus)+1))/2**discordant)
        compared.append({"candidate":candidate,"baseline":baseline,"better":plus,"worse":minus,
            "equal":int((delta==0).sum()),"EM_difference":delta.mean(),"paired_bootstrap_low":lo,"paired_bootstrap_high":hi,
            "exact_McNemar_p":p if protocol["settings"]["repeats"]==1 else np.nan})
    paired_table=pd.DataFrame(compared)
    display(paired_table.round(4)); paired_table.to_csv(ANALYSIS/"paired_answers.csv",index=False)
    display(Markdown("복수 개발 비교에 대한 p값은 탐색적 참고값이며 다중 비교 보정 또는 최종 검정을 대체하지 않는다."))


## 준비 비용·API 비용·실제 입력 규격

관계 추출은 공통 생성기와 같은 모델을 사용하되 **오프라인 준비용 호출**이고, 최종 답변의 공통 생성 설정을 바꾸지 않는다. 인용을 확인해도 관계의 의미가 올바르거나 모든 사실이 추출됐다는 보장은 없다. 서로 다른 이름의 동일 개체는 정확 문자열이 다르면 연결되지 않을 수 있다.


In [ ]:
if RUN_DIR is not None:
    prep=protocol["fact_index"]
    prep_fee=prep.get("api_cost_usd_known",prep["api_cost_usd"])
    if not prep.get("cost_complete",True):
        display(Markdown(f"준비 비용에는 중단된 요청 {prep['unresolved_http_requests']}건의 청구량이 미확인이다. 확인된 USD {prep_fee:.6f}와 아래 상각값은 하한이며 정확한 총액이 아니다."))
    display(pd.DataFrame([{k:prep[k] for k in ("passages","passages_with_facts","facts","rejected_facts","api_calls",
        "api_cost_usd","input_tokens","output_tokens","summed_request_seconds","new_wall_seconds")}]))
    fees=table[["retrieval_api_cost_usd","online_api_cost_usd","judgment_requests","decision_input_tokens",
        "llm_input_tokens","generation_evidence_tokens_proxy","evidence_count","input_truncated"]]
    display(fees.round(6)); fees.to_csv(ANALYSIS/"cost_and_inputs.csv")
    amortized=[]
    for q in [100,1000,10000,100000]:
        for m in ["C_plan","C_pack","D_pack","P_code"]:
            amortized.append({"queries_Q":q,"method":m,"prep_API_per_query":prep_fee/q,
                "online_API_per_query":table.loc[m,"online_api_cost_usd"],
                "amortized_API_per_query":prep_fee/q+table.loc[m,"online_api_cost_usd"]})
    display(pd.DataFrame(amortized).round(6))
    pd.DataFrame(amortized).to_csv(ANALYSIS/"preparation_amortization.csv",index=False)
    runtime=read_json(RUN_DIR/"runtime_after.json")["laya"]
    display(pd.DataFrame([runtime]).T)
    display(Markdown("API 요금은 기록된 가격 × 실제 usage의 추정 청구액이다. 로컬 GPU·전력·개발자 작업·자료 갱신 비용은 이 합계에 포함되지 않았다. 색인 준비의 요청 시간 합계는 병렬 실행의 벽시계 시간과 다르다."))


In [ ]:
if RUN_DIR is not None:
    paired_question=df[df.method=="C_pack"].sort_values(["complete_evidence_hit","answer_em"]).iloc[0].question_id
    detail=next(r for r in records if r["retrieval"]["question_id"]==paired_question and r["retrieval"]["method"]=="C_pack")
    print("점검 문항:",detail["retrieval"]["question"])
    print("선택 계획:",detail["retrieval"]["selected_plan_ids"])
    print("탐색 진단:",detail["retrieval"]["diagnostics"])
    print("답변:",detail["generation"]["answer"])
    for trace in detail["generation_view"].get("packing_trace",[]):
        print("\n원문",trace["passage_id"],trace["title"],"문단 내 문자 위치:",trace["passage_character_spans"])
        print("\n".join(trace["quotes"]))
    print("\n실제 LLM 프롬프트:\n",detail["generation"]["prompt"]["text"])


In [ ]:
if RUN_DIR is not None:
    model_inputs=[]
    for r in records:
        retrieval=r["retrieval"]
        mt=retrieval.get("model_tokens",{})
        model_inputs.append({"method":retrieval["method"],
            "query_encoder_nonpad_tokens":(mt.get("retriever_query") or {}).get("nonpad",0),
            "reranker_nonpad_tokens":(mt.get("reranker_pairs") or {}).get("nonpad",0),
            "query_truncated_sequences":mt.get("retriever_query",{}).get("truncated_sequences",0),
            "reranker_truncated_sequences":mt.get("reranker_pairs",{}).get("truncated_sequences",0),
            "Jev_or_Laya_input_tokens":retrieval.get("decision_usage",{}).get("input_tokens",0)})
    inputs=pd.DataFrame(model_inputs).groupby("method").mean().reindex(METHODS)
    display(inputs.round(2));inputs.to_csv(ANALYSIS/"model_input_components.csv")
    prep=protocol["fact_index"]
    for session,sha in prep.get("ledger_sha256",{}).items():
        assert file_sha256(ROOT/session/"api_calls.jsonl")==sha,"Preparation API ledger changed"
    print("준비 물리 HTTP 영수증 해시 검증:",len(prep.get("ledger_sha256",{})),"세션")
    print("이전 준비 실행의 관측 벽시계 시간(초):",prep.get("prior_preparation_wall_seconds",{}))
    display(Markdown("A/B의 판별 토큰 그래프 0은 별도 Jev/Laya 호출이 없다는 뜻이다. A/B 임베딩·재정렬 입력은 위 표와 원시 로컬 추론 로그에 별도로 남아 있다. 서로 다른 모델 토크나이저의 토큰을 합산하지 않는다."))

    display(Markdown("GPU 할당/최대 메모리는 이 프로세스의 A/B와 Laya 공동 로딩을 포함한다. Laya 단독 메모리로 해석하지 않는다."))


## 해석과 로그 위치

가설 전체를 한 번에 성공/실패로 판정하지 않는다. 후보 확보율이 낮으면 관계 추출·이름 연결·경로 범위를 먼저 검토한다. C가 코드 대조군보다 좋은 선택을 못하면 판단모델의 추가 역할은 확인되지 않은 것이다. 인용에서 EM이 낮아지면 입력을 줄인 것만으로 개선이라 하지 않는다. 같은 품질을 확보한 조건의 시간과 입력 비용을 비교한다.

- 본문·질문·원시 판단·실제 전달 근거·실제 GPT 요청/응답: 실행 폴더와 연결된 `results/raw/...`.
- 관계별 subject/relation/object, 원문 인용, 문단 내 문자 위치: `data/plan_index/.../facts.jsonl`.
- 준비 호출 원문·검증에서 거부된 관계·청구량: 같은 색인의 `batches.jsonl`과 `preparation.json`에 연결된 별도 원시 로그.
- CSV·그림·질문별 실패 구분: 실행 폴더의 `analysis/`.

설계의 출처는 [SUQL](https://aclanthology.org/2024.findings-naacl.283.pdf), [RoG](https://proceedings.iclr.cc/paper_files/paper/2024/file/3e2aeb66481dd63a32421bf032b70384-Paper-Conference.pdf), [LOTUS](https://www.vldb.org/pvldb/vol18/p4171-patel.pdf)의 아이디어를 참고한 **우리의 가설 구현**이다. 해당 논문이 이 새 조합이나 준비 오류 없는 검색을 보장한 것은 아니다.


In [ ]:
if RUN_DIR is not None:
    b,c,cp,d,p=(table.loc[m] for m in ["B","C_plan","C_pack","D_pack","P_code"])
    display(Markdown(f"**이번 {len(protocol['question_ids'])}개 개발 문항 관측:** "
        f"B / C 원문 / C 인용 / D 인용 / 코드 인용의 EM은 {b.answer_em:.1%} / {c.answer_em:.1%} / {cp.answer_em:.1%} / {d.answer_em:.1%} / {p.answer_em:.1%}. "
        f"C 후보 원문 확보율은 {cp.complete_candidate_hit:.1%}, 최종 원문 확보율은 {cp.complete_evidence_hit:.1%}. "
        f"C 인용의 GPT 입력은 {cp.llm_input_tokens:.0f}토큰, 전체 {cp.online_seconds:.2f}초이며 B는 {b.llm_input_tokens:.0f}토큰, {b.online_seconds:.2f}초다. "
        "이는 개발 관측이며 최종 우위 또는 원인 증명이 아니다."))
    plan_rows = df[df.method == "C_plan"]
    reachable = plan_rows[plan_rows.complete_candidate_hit == 1]
    display(Markdown(f"**다음 개선 대상:** 후보에 필요한 원문 ID가 모두 포함된 관측은 {len(reachable)}/{len(plan_rows)}개이고, "
        f"Jev는 그중 {int(reachable.complete_evidence_hit.sum())}개에서 필요한 원문 ID를 모두 반환했다. "
        "이는 후보에 진입한 경우의 선택 성능이며 전체 정확도가 아니다. 선택한 원문은 후보 원문의 부분집합이므로, "
        "후보 밖 근거는 판단모델만 교체해서 복구할 수 없다. 관계 추출·이름 연결·경로 후보의 누락을 먼저 검토해야 한다. "
        "정답 원문에 사실 하나가 있다는 사실만으로 필요한 관계까지 추출되었다고 볼 수 없다."))
    if PACKING_DIR is not None:
        reduction = 1 - packing_df.packed_input_tokens.mean() / packing_df.original_input_tokens.mean()
        display(Markdown(f"**동일 근거 품질 재생:** GPT 입력 {packing_df.original_input_tokens.mean():.1f} → "
            f"{packing_df.packed_input_tokens.mean():.1f}토큰({reduction:.1%} 감소), EM "
            f"{packing_df.original_em.mean():.1%} → {packing_df.packed_em.mean():.1%}. "
            f"좋아진 답변 {int((packing_df.packed_em > packing_df.original_em).sum())}개, "
            f"나빠진 답변 {int((packing_df.packed_em < packing_df.original_em).sum())}개다. "
            "집계가 같아도 인용이 무손실이라는 보장은 없고, 생성 자체의 변동도 배제하지 않는다. "
            "이 재생의 생성 시간은 실제 전체 지연 비교에 사용하지 않는다."))
    write_json(ANALYSIS/"validation_checks.json",{"completed_receipts_and_ledger":True,"paired_generation_observations":len(df),
        "reported_decision_truncations":int(df.input_truncated.sum()),"corpus_sha256":protocol["corpus_sha256"],
        "fact_index_sha256":protocol["fact_index"]["facts_sha256"],"development_only":True,
        "original_evidence_and_packing_verified":True,"A_B_same_candidates":True})
    write_json(ANALYSIS/"analysis_manifest.json",{"run":str(RUN_DIR),"system_sha256":file_sha256(RUN_DIR/"system.jsonl"),
        "notebook_source_sha256":notebook_source_sha256(ROOT/"notebook/09_test_plan_routing.ipynb"),
        "exports_sha256":{f.name:file_sha256(f) for f in ANALYSIS.iterdir() if f.is_file() and f.name!="analysis_manifest.json"}})
    print("질문별 결과:",RUN_DIR/"questions.csv")
    print("원시 결과:",RUN_DIR/"system.jsonl")
    print("온라인 원시 호출:",ROOT/manifest["raw_log"])
    print("준비 원시 호출:",protocol["fact_index"]["raw_log_sessions"])
    print("그림·표:",ANALYSIS)
